# Load deep-learning baselines (Darts)


In [1]:
# !pip install -r requirements/darts.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"load"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path:
    sys.path.insert(0,str(ROOT/"src"))

from sklearn import preprocessing

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel, TSMixerModel, RNNModel, TiDEModel, DLinearModel, NLinearModel
from darts.utils.likelihood_models import QuantileRegression

from preprocessing import load_data
from gefcom import (
    load_gefcom_load,
    evaluate_gefcom_load,
)
from src.metrics import print_metrics

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))

MODEL_CLASSES={
    "tft":TFTModel,
    "tsmixer":TSMixerModel,
    "deepar":RNNModel,
    "tide":TiDEModel,
    "dlinear":DLinearModel,
    "nlinear":NLinearModel,
}

OUTPUT_DIR=ROOT/"outputs"/"predictions"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)


## GEFCom2014-L


In [43]:
GEF_MODEL="deepar"  # tft, tsmixer, deepar, tide, dlinear, nlinear
GEF_LOAD_EXISTING=True
GEF_MODEL_DIR=ROOT/"models"/"darts"/"gefcom"
GEF_MODEL_DIR.mkdir(parents=True,exist_ok=True)
GEF_MODEL_PATH=GEF_MODEL_DIR/f"{GEF_MODEL}.darts"

gef_df=load_gefcom_load(ROOT/"data"/"gefcom_load_data.csv")

# Full target series and task-1 training history.
gef_series=TimeSeries.from_dataframe(
    gef_df,
    time_col="timestamp",
    value_cols="target",
    freq="h",
).astype(np.float32)

gef_task1_df=gef_df[gef_df["task_id"]==1].sort_values("timestamp").copy()
gef_task1=TimeSeries.from_dataframe(
    gef_task1_df,
    time_col="timestamp",
    value_cols="target",
    freq="h",
).astype(np.float32)

# Same scaling convention 
gef_train,gef_val=gef_task1.split_after(0.8)

gef_target_scale=Scaler(preprocessing.StandardScaler())
gef_train_t=gef_target_scale.fit_transform(gef_train)
gef_val_t=gef_target_scale.transform(gef_val)
gef_series_t=gef_target_scale.transform(gef_series)

print("GEF full target:",gef_series_t.start_time(),"->",gef_series_t.end_time())
print("GEF train:",gef_train.start_time(),"->",gef_train.end_time())
print("GEF validation:",gef_val.start_time(),"->",gef_val.end_time())

if GEF_LOAD_EXISTING:
    if not GEF_MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {GEF_MODEL_PATH}. "
            "Put the GEFCom-L checkpoint there or set GEF_LOAD_EXISTING=False."
        )
    gef_model=MODEL_CLASSES[GEF_MODEL].load(
        str(GEF_MODEL_PATH),
        weights_only=False,
    )
else:
    likelihood=QuantileRegression(BASE_QUANTILES)
    common=dict(
        input_chunk_length=168,
        output_chunk_length=24,
        likelihood=likelihood,
        n_epochs=100,
        random_state=42,
        force_reset=True,
    )

    if GEF_MODEL=="tft":
        gef_model=TFTModel(**common,hidden_size=64,lstm_layers=1,num_attention_heads=4)
    elif GEF_MODEL=="tsmixer":
        gef_model=TSMixerModel(**common,hidden_size=64)
    elif GEF_MODEL=="deepar":
        gef_model=RNNModel(
            model="LSTM",
            input_chunk_length=168,
            output_chunk_length=24,
            training_length=1000,
            hidden_dim=64,
            likelihood=likelihood,
            n_epochs=100,
            random_state=42,
            force_reset=True,
        )
    elif GEF_MODEL=="tide":
        gef_model=TiDEModel(**common,hidden_size=64)
    elif GEF_MODEL=="dlinear":
        gef_model=DLinearModel(**common)
    elif GEF_MODEL=="nlinear":
        gef_model=NLinearModel(**common,normalize=False)
    else:
        raise ValueError(GEF_MODEL)

    gef_model.fit(
        gef_train_t,
        val_series=gef_val_t,
        verbose=True,
    )
    gef_model.save(str(GEF_MODEL_PATH))


GEF full target: 2005-01-01 01:00:00 -> 2012-01-01 00:00:00
GEF train: 2005-01-01 01:00:00 -> 2009-08-07 05:00:00
GEF validation: 2009-08-07 06:00:00 -> 2010-10-01 00:00:00


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.15216736641670803 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


## PG&E competition


In [54]:
MODEL="nlinear"  # tft, tsmixer, deepar, tide, dlinear, nlinear
NON_GEF_LOAD_EXISTING=True
MODEL_DIR=ROOT/"models"/"darts"/"pge"
MODEL_DIR.mkdir(parents=True,exist_ok=True)
MODEL_PATH=MODEL_DIR/f"{MODEL}.darts"

data=load_data(ROOT/"data")

series=TimeSeries.from_dataframe(
    data.full,
    time_col="timestamp",
    value_cols="target",
    freq="h",
).astype(np.float32)

future_covariates=TimeSeries.from_dataframe(
    data.full,
    time_col="timestamp",
    value_cols=data.covariates,
    freq="h",
).astype(np.float32)

testing_cutoff=pd.Timestamp(data.context["timestamp"].max())

train,test=series.split_after(testing_cutoff)
train,val=train.split_after(0.8)

target_scale=Scaler(preprocessing.StandardScaler())
train_t=target_scale.fit_transform(train)
val_t=target_scale.transform(val)
test_t=target_scale.transform(test)
series_t=target_scale.transform(series)

future_train,future_test=future_covariates.split_after(testing_cutoff)
future_train,future_val=future_train.split_after(0.8)

future_scale=Scaler(preprocessing.StandardScaler())
future_train_t=future_scale.fit_transform(future_train)
future_val_t=future_scale.transform(future_val)
future_test_t=future_scale.transform(future_test)
future_cov_t=future_scale.transform(future_covariates)

print("Target:",series_t.start_time(),"->",series_t.end_time())
print("Future covariates:",future_cov_t.start_time(),"->",future_cov_t.end_time())
assert future_cov_t.end_time()>=series_t.end_time()

if NON_GEF_LOAD_EXISTING:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {MODEL_PATH}. "
            "Put the PG&E checkpoint there or set NON_GEF_LOAD_EXISTING=False."
        )
    model=MODEL_CLASSES[MODEL].load(
        str(MODEL_PATH),
        weights_only=False,
    )
else:
    likelihood=QuantileRegression(OUTPUT_QUANTILES)
    common=dict(
        input_chunk_length=168,
        output_chunk_length=24,
        likelihood=likelihood,
        n_epochs=100,
        random_state=42,
        force_reset=True,
    )

    if MODEL=="tft":
        model=TFTModel(**common,hidden_size=64,lstm_layers=1,num_attention_heads=4)
    elif MODEL=="tsmixer":
        model=TSMixerModel(**common,hidden_size=64)
    elif MODEL=="deepar":
        model=RNNModel(
            model="LSTM",
            input_chunk_length=168,
            output_chunk_length=24,
            training_length=1000,
            hidden_dim=64,
            likelihood=likelihood,
            n_epochs=100,
            random_state=42,
            force_reset=True,
        )
    elif MODEL=="tide":
        model=TiDEModel(**common,hidden_size=64)
    elif MODEL=="dlinear":
        model=DLinearModel(**common)
    elif MODEL=="nlinear":
        model=NLinearModel(**common,normalize=False)
    else:
        raise ValueError(MODEL)

    model.fit(
        train_t,
        val_series=val_t,
        future_covariates=future_train_t,
        val_future_covariates=future_val_t,
        verbose=True,
    )
    model.save(str(MODEL_PATH))


Target: 2020-01-01 00:00:00 -> 2022-12-31 23:00:00
Future covariates: 2020-01-01 00:00:00 -> 2022-12-31 23:00:00


In [55]:
backtest_series=model.historical_forecasts(
    series=series_t,
    future_covariates=future_cov_t,
    start=test.start_time(),
    forecast_horizon=24,
    stride=24,
    num_samples=100,
    last_points_only=False,
    retrain=False,
    verbose=True,
)

forecast_t=concatenate(backtest_series)
forecast=target_scale.inverse_transform(forecast_t)
gt=series.slice_intersect(forecast)
forecast=forecast.slice_intersect(gt)

result=pd.DataFrame({
    "item_id":1,
    "timestamp":gt.time_index,
    "target":gt.values().flatten(),
})
for q in OUTPUT_QUANTILES:
    result[str(q)]=forecast.quantile(q).values().flatten()
result["prediction"]=result["0.5"]

result.to_csv(
    OUTPUT_DIR/f"{MODEL}.csv",
    index=False,
)
print_metrics(result,BASE_QUANTILES,label=MODEL)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Generating TimeSeries: 100%|██████████| 365/365 [00:00<00:00, 10551.24it/s]


nlinear
MAE:  117.889511
RMSE: 163.527878
aQL:  47.058079


{'MAE': 117.88951110839844,
 'RMSE': 163.5278778076172,
 'aQL': 47.05807876586914}